بررسی ساختار دیتاست 


In [4]:
from huggingface_hub import HfApi
from huggingface_hub.hf_api import RepoFile, RepoFolder
from collections import Counter
from pathlib import PurePosixPath
import pandas as pd

# ==========================================
# 1. Dataset Configuration
# ==========================================

REPO_ID = "nexar-ai/nexar_collision_prediction"

api = HfApi()

# ==========================================
# 2. Get Repository Structure
# ==========================================

print("Fetching dataset structure...")

items = list(api.list_repo_tree(
    repo_id=REPO_ID,
    repo_type="dataset",
    recursive=True
))

# Separate files and folders
repo_files = [
    f for f in items
    if isinstance(f, RepoFile)
]

repo_folders = [
    f for f in items
    if isinstance(f, RepoFolder)
]

print("\n========== DATASET INFORMATION ==========")

print("Repository:", REPO_ID)
print("Total files:", len(repo_files))
print("Total folders:", len(repo_folders))

# ==========================================
# 3. Folder Structure
# ==========================================

print("\n========== FOLDER STRUCTURE ==========")

for folder in sorted(repo_folders, key=lambda x: x.path):
    depth = folder.path.count("/")

    if depth <= 2:
        print("  " * depth + folder.path)

# ==========================================
# 4. File Format Distribution
# ==========================================

extensions = Counter(
    PurePosixPath(f.path).suffix.lower()
    for f in repo_files
)

print("\n========== FILE FORMATS ==========")

for ext, count in sorted(extensions.items()):
    print(f"{ext or '[no extension]'}: {count}")

# ==========================================
# 5. Create File Inventory
# ==========================================

inventory = pd.DataFrame([
    {
        "path": f.path,
        "type": "file",
        "extension": PurePosixPath(f.path).suffix.lower(),
        "size_mb": (
            round(f.size / (1024**2), 2)
            if f.size is not None else None
        )
    }
    for f in repo_files
])

print("\n========== FILE INVENTORY ==========")

display(inventory.head(20))

# ==========================================
# 6. Video Distribution
# ==========================================

video_extensions = {".mp4", ".avi", ".mov", ".mkv"}

videos = inventory[
    inventory["extension"].isin(video_extensions)
].copy()

videos["folder"] = videos["path"].apply(
    lambda x: str(PurePosixPath(x).parent)
)

print("\n========== VIDEO DISTRIBUTION ==========")

video_distribution = (
    videos.groupby("folder")
    .size()
    .reset_index(name="video_count")
)

display(video_distribution)

print("Total videos:", len(videos))

# ==========================================
# 7. Dataset Size
# ==========================================

print("\n========== DATASET SIZE ==========")

total_size_gb = inventory["size_mb"].sum() / 1024
video_size_gb = videos["size_mb"].sum() / 1024

print(f"Total dataset size: {total_size_gb:.2f} GB")
print(f"Total video size: {video_size_gb:.2f} GB")

unknown_sizes = inventory["size_mb"].isna().sum()

if unknown_sizes:
    print(f"Warning: {unknown_sizes} files have unknown sizes.")

# ==========================================
# 8. Metadata Files
# ==========================================

print("\n========== METADATA FILES ==========")

metadata_files = inventory[
    inventory["path"].str.contains(
        r"metadata|\.csv$",
        case=False,
        regex=True
    )
]

display(metadata_files)

# ==========================================
# 9. Sample Video Files
# ==========================================

print("\n========== SAMPLE VIDEOS ==========")

display(
    videos[["path", "size_mb", "folder"]].head(10)
)

# ==========================================
# 10. Summary
# ==========================================

print("\n========== FINAL SUMMARY ==========")

print("Total files:", len(repo_files))
print("Total folders:", len(repo_folders))
print("Total videos:", len(videos))
print("Metadata/CSV files:", len(metadata_files))
print("Dataset size (GB):", round(total_size_gb, 2))

print("\n========== INSPECTION COMPLETED ==========")

Fetching dataset structure...

========== DATASET INFORMATION ==========
Repository: nexar-ai/nexar_collision_prediction
Total files: 2857
Total folders: 9

========== FOLDER STRUCTURE ==========
test-private
  test-private/negative
  test-private/positive
test-public
  test-public/negative
  test-public/positive
train
  train/negative
  train/positive

========== FILE FORMATS ==========
[no extension]: 2
.csv: 9
.md: 1
.mp4: 2844
.py: 1

========== FILE INVENTORY ==========


,path,type,extension,size_mb
0,.gitattributes,file,,0.00
1,LICENSE,file,,0.00
2,README.md,file,.md,0.00
3,evaluate_submission.py,file,.py,0.00
4,sample_submission.csv,file,.csv,0.03
5,solution.csv,file,.csv,0.02
6,test-private/negative/01047.mp4,file,.mp4,2.60
7,test-private/negative/01053.mp4,file,.mp4,4.79
8,test-private/negative/01055.mp4,file,.mp4,5.33
9,test-private/negative/01057.mp4,file,.mp4,5.33



========== VIDEO DISTRIBUTION ==========


,folder,video_count
0,test-private/negative,339
1,test-private/positive,338
2,test-public/negative,333
3,test-public/positive,334
4,train/negative,750
5,train/positive,750


Total videos: 2844

========== DATASET SIZE ==========
Total dataset size: 29.22 GB
Total video size: 29.22 GB

========== METADATA FILES ==========


,path,type,extension,size_mb
4,sample_submission.csv,file,.csv,0.03
5,solution.csv,file,.csv,0.02
345,test-private/negative/metadata.csv,file,.csv,0.01
684,test-private/positive/metadata.csv,file,.csv,0.02
1018,test-public/negative/metadata.csv,file,.csv,0.01
1353,test-public/positive/metadata.csv,file,.csv,0.02
1354,time_to_accident_test_map.csv,file,.csv,0.01
2105,train/negative/metadata.csv,file,.csv,0.02
2856,train/positive/metadata.csv,file,.csv,0.03



========== SAMPLE VIDEOS ==========


,path,size_mb,folder
6,test-private/negative/01047.mp4,2.60,test-private/negative
7,test-private/negative/01053.mp4,4.79,test-private/negative
8,test-private/negative/01055.mp4,5.33,test-private/negative
9,test-private/negative/01057.mp4,5.33,test-private/negative
10,test-private/negative/01063.mp4,5.29,test-private/negative
11,test-private/negative/01064.mp4,3.31,test-private/negative
12,test-private/negative/01077.mp4,2.36,test-private/negative
13,test-private/negative/01082.mp4,3.30,test-private/negative
14,test-private/negative/01086.mp4,2.89,test-private/negative
15,test-private/negative/01087.mp4,2.20,test-private/negative



========== FINAL SUMMARY ==========
Total files: 2857
Total folders: 9
Total videos: 2844
Metadata/CSV files: 9
Dataset size (GB): 29.22

========== INSPECTION COMPLETED ==========


دریافت متادیتا 

In [7]:
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import pandas as pd

# 1. Configuration
REPO_ID = "nexar-ai/nexar_collision_prediction"
SAVE_DIR = Path("data/metadata")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()

# 2. Find metadata files
repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

metadata_files = [
    path for path in repo_files
    if Path(path).name.lower() == "metadata.csv"
]

print("Metadata files found:", len(metadata_files))
print(*metadata_files, sep="\n")

# 3. Download and read metadata
metadata_dfs = {}

for file_path in metadata_files:

    local_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=file_path,
        local_dir=SAVE_DIR
    )

    df = pd.read_csv(local_path)
    metadata_dfs[file_path] = df

    print(f"\n========== {file_path} ==========")

    print("Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nFirst 5 Rows:")
    display(df.head())

print("\nMetadata download completed!")
print("Saved in:", SAVE_DIR.resolve())

Metadata files found: 6
test-private/negative/metadata.csv
test-private/positive/metadata.csv
test-public/negative/metadata.csv
test-public/positive/metadata.csv
train/negative/metadata.csv
train/positive/metadata.csv

========== test-private/negative/metadata.csv ==========
Shape: (339, 7)

Columns: ['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident']

Data Types:
file_name               str
time_of_event       float64
time_of_alert       float64
light_conditions        str
weather                 str
scene                   str
time_to_accident    float64
dtype: object

Missing Values:
file_name             0
time_of_event       339
time_of_alert       339
light_conditions      0
weather               0
scene                 0
time_to_accident      0
dtype: int64

First 5 Rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,01047.mp4,NaN,NaN,Normal,Clear,Urban,0.5
1,01053.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5
2,01055.mp4,NaN,NaN,Normal,Cloudy,Highway,0.5
3,01057.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5
4,01063.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5



========== test-private/positive/metadata.csv ==========
Shape: (338, 7)

Columns: ['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident']

Data Types:
file_name               str
time_of_event       float64
time_of_alert       float64
light_conditions        str
weather                 str
scene                   str
time_to_accident    float64
dtype: object

Missing Values:
file_name           0
time_of_event       0
time_of_alert       0
light_conditions    0
weather             0
scene               0
time_to_accident    0
dtype: int64

First 5 Rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,00001.mp4,19.967,18.067,Normal,Clear,Highway,0.5
1,00011.mp4,20.867,19.367,Normal,Clear,Highway,0.5
2,00012.mp4,18.722,18.461,Normal,Clear,Urban,0.5
3,00033.mp4,18.933,16.000,Normal,Clear,Highway,0.5
4,00034.mp4,20.067,18.633,Normal,Clear,Urban,0.5



========== test-public/negative/metadata.csv ==========
Shape: (333, 7)

Columns: ['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident']

Data Types:
file_name               str
time_of_event       float64
time_of_alert       float64
light_conditions        str
weather                 str
scene                   str
time_to_accident    float64
dtype: object

Missing Values:
file_name             0
time_of_event       333
time_of_alert       333
light_conditions      0
weather               0
scene                 0
time_to_accident      0
dtype: int64

First 5 Rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,01044.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5
1,01056.mp4,NaN,NaN,Normal,Clear,Urban,0.5
2,01059.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5
3,01065.mp4,NaN,NaN,Normal,Clear,Sub-urban,0.5
4,01068.mp4,NaN,NaN,Normal,Cloudy,Urban,0.5


... (3 outputs left out)

,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,01924.mp4,NaN,NaN,Normal,Clear,Sub-urban,NaN
1,01429.mp4,NaN,NaN,Normal,Rain,Urban,NaN
2,01904.mp4,NaN,NaN,Normal,Cloudy,Urban,NaN
3,01486.mp4,NaN,NaN,Normal,Clear,Urban,NaN
4,01171.mp4,NaN,NaN,Normal,Clear,Sub-urban,NaN



========== train/positive/metadata.csv ==========
Shape: (750, 7)

Columns: ['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident']

Data Types:
file_name               str
time_of_event       float64
time_of_alert       float64
light_conditions        str
weather                 str
scene                   str
time_to_accident    float64
dtype: object

Missing Values:
file_name             0
time_of_event         0
time_of_alert         0
light_conditions      0
weather               1
scene                 0
time_to_accident    750
dtype: int64

First 5 Rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident
0,00822.mp4,19.500,18.633,Normal,Cloudy,Urban,NaN
1,00208.mp4,19.800,19.233,Normal,Cloudy,Sub-urban,NaN
2,00072.mp4,20.101,20.068,Normal,Clear,Urban,NaN
3,00128.mp4,19.267,18.333,Normal,Clear,Urban,NaN
4,00205.mp4,21.146,18.196,Normal,Clear,Highway,NaN



Metadata download completed!
Saved in: C:\Users\ASUS\DL-Project\image_neural_network\data\metadata


جلوگیری از نشت دیتا ، شماره 1 


In [9]:
import pandas as pd

# Columns that may cause Data Leakage
leakage_columns = [
    "time_of_event",
    "time_of_alert",
    "time_to_accident"
]

prepared_metadata = {}

for file_path, original_df in metadata_dfs.items():

    # Keep original data unchanged
    df = original_df.copy()

    # Create binary labels
    df["label"] = df["time_of_event"].notna().astype(int)

    # Remove leakage columns
    df = df.drop(columns=leakage_columns)

    # Store processed metadata
    prepared_metadata[file_path] = df

    print(f"\n{file_path}")
    print("Shape:", df.shape)
    print("Labels:", df["label"].value_counts().to_dict())

# Check final structure
print("\n========== FINAL COLUMNS ==========")
display(prepared_metadata["train/positive/metadata.csv"].head())

# Verify removed columns
for df in prepared_metadata.values():
    assert not any(col in df.columns for col in leakage_columns)

print("\nData Leakage columns removed successfully!- S1")


test-private/negative/metadata.csv
Shape: (339, 5)
Labels: {0: 339}

test-private/positive/metadata.csv
Shape: (338, 5)
Labels: {1: 338}

test-public/negative/metadata.csv
Shape: (333, 5)
Labels: {0: 333}

test-public/positive/metadata.csv
Shape: (334, 5)
Labels: {1: 334}

train/negative/metadata.csv
Shape: (750, 5)
Labels: {0: 750}

train/positive/metadata.csv
Shape: (750, 5)
Labels: {1: 750}

========== FINAL COLUMNS ==========


,file_name,light_conditions,weather,scene,label
0,00822.mp4,Normal,Cloudy,Urban,1
1,00208.mp4,Normal,Cloudy,Sub-urban,1
2,00072.mp4,Normal,Clear,Urban,1
3,00128.mp4,Normal,Clear,Urban,1
4,00205.mp4,Normal,Clear,Highway,1



Data Leakage columns removed successfully!- S1
